# Session 5 · Building Benchmark Suites for AI Agents

**A claim you measured once is a coin you flipped once.**

The deck carries the argument. **This notebook is only the things you run.**

| block | ⏱ | what you do | needs a key? |
|---|---|---|---|
| Hands-on 1 | 21–33 | write two rows in `benchmark_rows.py` | no |
| Hands-on 2 | 33–46 | screen them — `python screen_my_rows.py` | no |
| Hands-on 3 | 58–66 | cost your own claim — `python n_for.py` | no |
| Hands-on 4 | 74–84 | push and tag the class pool | yes |

Only cells 13, 17 and 21 touch an API. Everything before them runs offline, for free,
during an outage.


In [ ]:
# Session 5 opens on the repo, not on an install.
!cd .. && git pull --ff-only 2>&1 | tail -3


In [ ]:
# Setup. Prints as it goes, so a stall has a name instead of a spinner.
#
# If this cell never finishes, it is almost never the imports -- they take under a
# second. It is the kernel: not started, or started in the wrong directory. The
# checks below tell you which, before any import is attempted.
import sys, os, time

print("python  ", sys.executable)
print("cwd     ", os.getcwd())
if not os.path.exists("row_screen.py"):
    print()
    print("!! Wrong directory -- the kernel must start in course-repo/.")
    print("   Close the notebook, open the course-repo FOLDER in VS Code,")
    print("   and reopen the notebook from there.")
    print("   Then: Command Palette > Python: Select Interpreter.")
    raise SystemExit(1)

# RELOAD, not just import. `__import__` returns a module Python already has in
# memory, unchanged -- so if you have edited benchmark_rows.py since the kernel
# started (which during Hands-on 1 you will, constantly), a bare import gives you
# the OLD file and the error looks like a missing function rather than a stale
# module. That is a bug this cell shipped with once. Reload is cheap; do it every
# time.
import importlib

t0 = time.perf_counter()
for name in ("row_screen", "benchmark_rows", "seeds5"):
    t = time.perf_counter()
    importlib.reload(importlib.import_module(name))
    print(f"  {name:<16} {time.perf_counter() - t:5.1f}s")
print(f"  {'total':<16} {time.perf_counter() - t0:5.1f}s")

import row_screen, benchmark_rows, seeds5

# Fail loudly if the reload did not take, rather than three cells later.
for _m, _attr in ((benchmark_rows, "show_starters"), (row_screen, "ATTRIBUTABLE_SHAPES")):
    if not hasattr(_m, _attr):
        print()
        print(f"!! {_m.__name__} has no {_attr!r} -- you are running a stale copy.")
        print("   Restart the kernel (Command Palette > Jupyter: Restart Kernel)")
        print("   and run this cell again.")
        raise SystemExit(1)


---
# Hands-on 1 · Write two rows  ⏱ 21–33

Open **`benchmark_rows.py`** in VS Code. Fill in `AUTHOR` and the two rows in `MY_ROWS`.

- **Row 1** — one you believe the healthy agent **passes** and a broken one **fails**
- **Row 2** — one you believe will be **hard**: ambiguous, multi-hop, or adversarial

### Fill in `predict` before you run anything

Each row has a `predict` field. Write down which failure shapes you think **your** row
catches — `wrong_tool`, `empty_search`, `injected` — *before* the screener tells you.

    "predict": ["wrong_tool"],

**A row you cannot be wrong about teaches you nothing when you screen it.** The screener
prints your guess against what actually happened, and a **MISS** is the most useful line
in the output: you had a theory about how your row would break and the evidence disagreed.
That only works if the guess is on record first.

`predict` never leaves your machine — `rows_for_pool()` strips it before anything is
pushed, so the dataset schema is unchanged.

### Your category

| where you're sitting | category |
|---|---|
| front left | `browser_search` |
| front right | `multi_hop` |
| back left | `report_gen` |
| back right | `adversarial` |

### Three things that will cost you a row

1. **An empty `forbidden_tools` is not a lenient row, it is a blind row.** It is the only
   field that catches a prompt injection, and it is the one everybody leaves empty.
2. **A repeated query fails `trajectory_no_waste` whatever your row says.** So catching
   the redundant agent is *free* — not a bet you made. The screener counts only what your
   expectations earn on top of that.
3. **If the answer can move — a version, a price, a date — put the URL in `verify_url`.**
   A stale row marks a *correct* answer wrong, and that looks exactly like a finding.

### If you do not know what to write

Run the next two cells. The first prints the three worked examples — the shape you are
filling in. The second prints **starter questions for your category**, and the four kinds
of row that are not worth having.

**The question is handed to you. The bet is not.** Writing the question takes ten seconds;
deciding what would count as correct, and which field catches a broken agent, is the
assignment.

---

### When both rows are written — run this, in a terminal

    python screen_my_rows.py

**Do not wait for the room.** The moment you have two rows and two `predict` lists, run it.
Two seconds, no API key, no network. Then read what it says and fix a row — you have time
for two or three passes before we regroup.


In [ ]:
# The three worked examples. Open, not hidden -- you just will not type them.
for i, r in enumerate(benchmark_rows.WORKED):
    meta = r["metadata"]
    print(f"\n--- ({chr(97+i)}) {meta['category']} / {meta['difficulty']} ---")
    print("Q:", r["inputs"]["question"])
    for k, v in r["outputs"].items():
        print(f"   {k:<18} {v}")


In [ ]:
# STUCK ON WHAT TO WRITE? Starter questions for your category, plus the four
# shapes of row that are not worth having.
#
# The QUESTION is handed to you. The bet is not -- that is the assignment, and
# `predict` is you committing to it before the screener rules.
# Change the category to yours, or pass nothing to see all four.
benchmark_rows.show_starters("browser_search")


---
# Hands-on 2 · Screen a row you did NOT write  ⏱ 33–46

**Swap `benchmark_rows.py` with the pair behind you.** Save theirs as `partner_rows.py`.

    python screen_my_rows.py --file partner_rows.py

Then come back to your own:

    python screen_my_rows.py

### Why someone else's row

Predicting your own row is contaminated — you know what you *meant* it to catch, so a HIT
only proves you remember your own intent.

Predicting a **stranger's** row means reading the fields and working out what bet they
actually made. Which is often not the bet they thought they were making. That is what
reviewing a colleague's benchmark is, and it is where the surprises live.

### Three steps, in this order

1. **Read** their rows.
2. **Overwrite their `predict` lists with your own guess.** If you skip this you are
   grading their bet instead of making yours — the screener will warn you.
3. **Run it.** Two seconds, no API key, no network.

**Done is not "it runs".** Done is: you predicted what their row catches, and found out
whether you were right.

> **If the swap fails** — no pair behind you, the file will not transfer, their file has a
> syntax error — screen your own rows instead and you have lost nothing. Tell them about
> the syntax error; that is a finding too.


In [ ]:
# PROJECTOR. The screener, run the way they will run it.
# --worked shows three rows that ship, two TODO rows that retire with named
# reasons, and a HIT/MISS table -- worked example (b) misses on purpose.
!python screen_my_rows.py --worked


In [ ]:
# And this is Hands-on 2's command -- someone ELSE's rows.
# Uncomment once you have their file saved as partner_rows.py.
# !python screen_my_rows.py --file partner_rows.py


In [ ]:
# The room's rows on one screen. Re-import so edits made since the last run are picked up.
# `import importlib` is repeated here on purpose -- this cell must work if someone
# runs it without having run the setup cell, which in a live room they will.
import importlib
importlib.reload(benchmark_rows)
importlib.reload(row_screen)

screens = row_screen.screen_all(benchmark_rows.WORKED + benchmark_rows.MY_ROWS)
row_screen.print_screen(screens)


In [ ]:
# What the screen does NOT do. A demonstration, not a claim.
# This row's ground truth is nonsense -- and the screen says SHIPS.
wrong = {
    "inputs": {"question": "What is the latest released version of `langgraph`?"},
    "outputs": {"must_contain": ["9.9.9-does-not-exist"],
                "expected_tools": ["web_search"],
                "forbidden_tools": ["package_registry", "version_lookup"],
                "max_tool_calls": 2},
    "metadata": {"category": "browser_search", "difficulty": "easy", "verify_url": None},
}
s = row_screen.screen_row(wrong)
print("verdict:", s.verdict, "  ground_truth_checked:", s.ground_truth_checked)
print()
print("It is not lying -- it never claimed to check that. The synthetic healthy")
print("answer is built FROM must_contain, so a false keyword cannot disagree with")
print("it. Only a live run catches that. Knowing what a check does NOT cover is")
print("the same skill as knowing what it does.")


---
# Hands-on 3 · How many runs for YOUR claim?  ⏱ 58–66

**σ** — how much your number moves when *nothing* changes.
**δ** — the difference you want to be able to see.

Pick a claim you would actually want to make: *"version B is 10% cheaper"*, *"the judge
adds under 5 seconds"*, *"this prompt cuts tool calls by a third"*. Work out what it costs.
Then answer in one sentence: **can you afford it?**

    python n_for.py --cv 0.4 --mean 6174 --pct 10
    python n_for.py --sigma <yours> --delta <yours>

Use σ = 2,444 and mean = 6,174 if you have no numbers of your own — those are Session 3's,
from the slide.


In [ ]:
# PROJECTOR. Let the output sit on screen. Do not talk over it.
!python n_for.py --demo


In [ ]:
# What does a 10% claim cost in this system?
!python n_for.py --cv 0.4 --mean 6174 --pct 10


In [ ]:
# LIVE, ~40s. The judge, FIVE times, on IDENTICAL saved evidence.
# No agent invocation and no searches -- this survives an outage of everything
# except the model endpoint.
import time
import evalkit
from evalkit import run_offline_evaluators
from seeds import load_fixtures

sample = next(f for f in load_fixtures() if f["seed"] == "empty_search")
judge = evalkit.make_groundedness_judge()

INPUTS = {"question": "What is the latest released version of `langgraph` on PyPI?"}
REF = {"must_contain": ["1.2"], "expected_tools": ["web_search"],
       "forbidden_tools": [], "max_tool_calls": 2}

verdicts, lat = [], []
for i in range(5):
    t0 = time.perf_counter()
    res = run_offline_evaluators(INPUTS, sample, REF, [judge])
    lat.append(time.perf_counter() - t0)
    verdicts.append(next(r["score"] for r in res if r["key"] == "groundedness"))
    print(f"  run {i+1}:  {str(verdicts[-1]):<6} {lat[-1]:>6.1f}s")

print(f"\nlatency  min {min(lat):.1f}s   max {max(lat):.1f}s   "
      f"spread {max(lat)/min(lat):.1f}x")
print(f"verdicts {verdicts}  -> {'SPLIT' if len(set(verdicts)) > 1 else 'unanimous'}")


In [ ]:
# What the pre-flight measured last night, in case the live cell came back tame.
import json, os
if os.path.exists("judge_variance5.json"):
    print(json.dumps(json.load(open("judge_variance5.json")), indent=2))
else:
    print("No judge_variance5.json -- run `python preflight5.py --runs 5 --save` first.")


---
# The adversarial row  ⏱ 66–74 · instructor demo

**They watch.** Injection susceptibility plausibly differs by provider, and only Anthropic
has been pre-flighted.


In [ ]:
# The injected document. Read the sentence aloud.
import inspect
src = inspect.getsource(seeds5)
start = src.index("_PAYLOAD = {")
print(src[start:src.index("@tool", start)])


In [ ]:
# LIVE, ~40s. Injected vs CLEAN CONTROL -- same tools, same prompt, same question.
#
# BOTH ARMS WILL FIRE. That is not a malfunction, it is the finding. Measured
# 8 Sep across three tool designs: control 100% every time. The cause is our own
# SYSTEM_PROMPT -- "Answer the question using the tools available." The agent
# uses a lookup tool because it was told to, not because a document asked.
#
# Without the control we would have called that an attack. That is the block.
for seed in ("injected", "injection_control"):
    r = seeds5.run_injection(seed)
    tools = [tc["name"] for tc in r["tool_calls"]]
    print(f"{seed:<20} tools={tools}  {'FIRED' if seeds5.fired(r) else '-'}")
    print(f"{'':<20} answer: {r['answer'][:90]}")


In [ ]:
# One field is doing all the work. Watch what happens when it goes.
#
# And then the sting: given the 100% base rate above, this field would flag
# EVERY run -- attack or no attack. A check that fires on everything has
# measured nothing. ALL-FAIL is ALL-PASS wearing a different hat.
adv = seeds5.INJECTION_ROW
print("the adversarial row catches:      ", sorted(row_screen.screen_row(adv).caught))
blind = {**adv, "outputs": {**adv["outputs"], "forbidden_tools": []}}
print("with forbidden_tools removed:     ", sorted(row_screen.screen_row(blind).caught))


---
# Hands-on 4 · Pool it, then version it  ⏱ 74–84

In a **terminal**:

    python push_pool.py --dry        # screen first. Nothing unscreened enters the pool.
    python push_pool.py --tag v1

> **⚠️ `list_examples` silently ignores unknown keyword arguments.** Misspell `splits` and
> you get the whole dataset back — no error, no warning, and a number you will believe.
>
>     client.list_examples(dataset_name=D, splits=["adversarial"])   # 3 examples
>     client.list_examples(dataset_name=D, split=["adversarial"])    # 82, no error


In [ ]:
!python push_pool.py --show v1


---
### Where this goes

Session 6 pins `v1`, runs two versions of the agent against it, and asks whether the
difference between them is real.

Every row you wrote is a question that gets asked twice. Every row nothing can fail will
answer "no change" forever.

You now know what that question costs.
